# UTS Data Mining II — IMAGE PROCESSING & K-MEANS

Dua contoh: **(A) clustering gambar angka** menggunakan dataset bawaan `load_digits()` dan **(B) segmentasi warna** menggunakan foto bawaan `skimage.data`. Keduanya tidak perlu mengunduh dataset gambar dari web.

**Petunjuk:** Buka di Google Colab → `Runtime` → `Run all`, atau jalankan dari atas secara berurutan. Setiap sel **kode** menghasilkan output. Semua angka hasil yang dibahas pada catatan setelah sel adalah panduan interpretasi, **bukan hasil yang diasumsikan sudah diperoleh**.

## Panduan awal — membuka berbagai file

| Jenis file | Perintah paling singkat | Keterangan |
|---|---|---|
| CSV | `pd.read_csv('file.csv')` | Tabel dengan pemisah koma |
| TSV | `pd.read_csv('file.tsv', sep='\t')` | Tabel dengan pemisah tab |
| Excel | `pd.read_excel('file.xlsx')` | Mungkin perlu `openpyxl` |
| JSON | `pd.read_json('file.json')` | Data JSON yang cocok untuk tabel |
| TXT | `open('file.txt', encoding='utf-8').read()` | Teks bebas; tidak harus melalui pandas |
| Gambar | `PIL.Image.open('foto.jpg')` atau `cv2.imread(...)` | **Bukan** dibuka langsung dengan pandas |
| Audio | `librosa.load('audio.wav')` atau `sf.read(...)` | **Bukan** dibuka langsung dengan pandas |

Pandas terutama dipakai untuk data berbentuk tabel dan untuk menampung fitur yang sudah diekstraksi dari teks/gambar/audio. Dataset latihan di notebook ini diakses langsung tanpa perlu upload manual (koneksi internet diperlukan untuk unduhan pertama).

## Contoh tabel pandas

In [ ]:
import pandas as pd
from IPython.display import display

# Contoh objek DataFrame; .head() untuk melihat baris pertama.
contoh = pd.DataFrame({'jenis': ['Text', 'Image', 'Audio'],
                       'fitur': ['TF-IDF', 'Pixel', 'MFCC']})
display(contoh)
print('Ukuran tabel:', contoh.shape)

**Interpretasi:** Output menunjukkan tiga jenis data beserta contoh fitur numeriknya. Untuk pemeriksaan tabel gunakan `.head()`, `.shape`, `.info()`, `.isna().sum()`.

**Keyword:** `pandas read_csv sep tab`, `pandas head shape info`, `read_excel google colab`.

## 1. Import library

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from skimage import data
print('NumPy = array, matplotlib = plot, sklearn = data + clustering, skimage = contoh foto.')

**Interpretasi:** `from sklearn.datasets import load_digits` mengambil fungsi tertentu dari library utama **scikit-learn**. `from skimage import data` adalah kumpulan contoh gambar. **Keyword:** `sklearn datasets load_digits`, `skimage data astronaut`, `python from import explanation`.

## 2. Dataset gambar angka dan metadata

In [ ]:
digit = load_digits()
gambar = digit.images
print('Total data dan resolusi per gambar:', gambar.shape)
print('Tipe array:', gambar.dtype)
print('Rentang intensitas pixel:', gambar.min(), 'sampai', gambar.max())
print('Contoh label digit (untuk referensi):', digit.target[:12])
plt.figure(figsize=(3, 3))
plt.imshow(gambar[0], cmap='gray')
plt.title('Contoh digit 8 x 8 (grayscale)')
plt.axis('off')
plt.show()

**Interpretasi:** Shape `(jumlah gambar, tinggi, lebar)`; setiap gambar berukuran **8 × 8 pixel** dan satu kanal grayscale. Intensitas pada `load_digits` adalah **0–16**, bukan rentang 8-bit 0–255. **Keyword:** `numpy image shape dtype min max`, `grayscale image 8x8 sklearn digits`, `image resolution pixel channel`.

## 3. Sampling 500 gambar

In [ ]:
n_sampel = 500  # Bisa diganti 100, 300, 1000 (asal <= jumlah gambar).
rng = np.random.default_rng(42)
indeks = rng.choice(len(gambar), size=min(n_sampel, len(gambar)), replace=False)
gambar_sample = gambar[indeks]
label_asli = digit.target[indeks]  # Hanya untuk pemeriksaan; tidak dipakai K-Means.
print('Shape sebelum sampling:', gambar.shape)
print('Shape setelah sampling:', gambar_sample.shape)
print('Shape label referensi:', label_asli.shape)

**Interpretasi:** Sampling diambil per **gambar**, tanpa pengembalian (`replace=False`). Setiap data yang dipilih masih berbentuk matriks **8×8**. **Keyword:** `numpy random choice replace False`, `random sampling images numpy`, `image array shape after sampling`.

## 4. Flatten: gambar 2D menjadi vektor 1D

In [ ]:
X_digit = gambar_sample.reshape(len(gambar_sample), -1)
print('Sebelum flatten:', gambar_sample.shape)
print('Sesudah flatten:', X_digit.shape)
print('Contoh fitur gambar pertama (64 pixel):')
print(X_digit[0])

**Interpretasi:** Format masukan K-Means = `(n_samples, n_features)`. Bentuk `(500, 8, 8)` menjadi `(500, 64)` karena **8×8 = 64 pixel per gambar**. `-1` membuat NumPy menghitung otomatis panjang vektornya. **Keyword:** `numpy reshape n_samples -1 image flatten`, `ValueError Found array with dim 3 KMeans`, `reshape 2D image for machine learning`.

## 5. Clustering K-Means gambar angka

In [ ]:
km_digit = KMeans(n_clusters=10, random_state=42, n_init=10)
cluster_digit = km_digit.fit_predict(X_digit)
print('Shape data yang masuk K-Means:', X_digit.shape)
print('Shape hasil label cluster:', cluster_digit.shape)
print('Jumlah data per cluster:')
print(np.bincount(cluster_digit, minlength=10))
print('Silhouette:', round(silhouette_score(X_digit, cluster_digit), 3))

**Interpretasi:** `n_clusters=10` dipilih sebagai latihan karena ada 10 jenis digit (0–9), **bukan** berarti K-Means tahu identitas digit tersebut. Silhouette mengukur pemisahan cluster; mendekati 1 lebih baik. **Keyword:** `KMeans images digits dataset`, `sklearn silhouette_score image features`, `kmeans cluster sizes np bincount`.

## 6. Visualisasi sampel digit per cluster

In [ ]:
fig, ax = plt.subplots(2, 5, figsize=(11, 5))
for c in range(10):
    idx = np.flatnonzero(cluster_digit == c)[0]
    ax.flat[c].imshow(gambar_sample[idx], cmap='gray')
    ax.flat[c].set_title(f'Cluster {c}')
    ax.flat[c].axis('off')
plt.tight_layout()
plt.show()
print('Masing-masing panel menunjukkan satu contoh data dari cluster terkait.')

**Interpretasi:** Tampilan hanya mengambil **satu contoh anggota tiap cluster**, bukan pusat cluster. Bandingkan bentuk karakter antarcluster; satu digit asli bisa tersebar pada beberapa cluster. **Keyword:** `matplotlib imshow subplots kmeans image`, `numpy flatnonzero cluster indices`.

## 7. Visualisasi cluster melalui PCA

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_digit)
print('Shape sebelum PCA:', X_digit.shape)
print('Shape setelah PCA:', X_pca.shape)
plt.figure(figsize=(7, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_digit, cmap='tab10', s=16)
plt.title('Clustering gambar digit — PCA 2D')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.colorbar(label='Nomor cluster')
plt.show()

**Interpretasi:** PCA mengubah **64 fitur pixel menjadi 2 komponen** untuk plotting. Clustering tetap menggunakan data 64 fitur sebelumnya. PCA tidak sama dengan flatten. **Keyword:** `PCA(n_components=2) image clustering`, `PCA plot KMeans digits`, `difference flatten and PCA`.

## 8. Contoh gambar RGB dan informasi resolusi

In [ ]:
rgb = data.astronaut()
print('Shape awal gambar RGB:', rgb.shape)
print('Resolusi (tinggi x lebar):', rgb.shape[0], 'x', rgb.shape[1])
print('Jumlah channel:', rgb.shape[2])
print('Tipe pixel:', rgb.dtype)
print('Rentang nilai:', rgb.min(), rgb.max())
plt.figure(figsize=(5, 4))
plt.imshow(rgb)
plt.title('Foto RGB - skimage.data.astronaut()')
plt.axis('off')
plt.show()

**Interpretasi:** Gambar warna berupa `(tinggi, lebar, 3)` dengan kanal **R, G, B**. Nilai pixel `uint8` lazimnya 0–255. RGB dari scikit-image berbeda dengan default BGR saat dibaca dengan OpenCV (`cv2.imread`). **Keyword:** `skimage.data.astronaut RGB shape`, `OpenCV BGR vs RGB`, `numpy image metadata channels`.

## 9. Flatten semua pixel dan sampling 1000 pixel

In [ ]:
pixel_flat = rgb.reshape(-1, 3)
print('Shape sebelum flatten:', rgb.shape)
print('Shape setelah flatten:', pixel_flat.shape)

indeks_pixel = np.random.default_rng(42).choice(len(pixel_flat), size=1000, replace=False)
X_pixel = pixel_flat[indeks_pixel]
print('Shape setelah sampling 1000 pixel:', X_pixel.shape)
print('Contoh 5 pixel RGB:')
print(X_pixel[:5])

**Interpretasi:** Kini **unit data = satu pixel**, bukan satu foto. `(H,W,3)` diubah menjadi `(H×W,3)`; tiga fiturnya adalah R,G,B. Jangan memakai `reshape(n_gambar,-1)` untuk kasus segmentasi warna ini karena unit sampelnya berbeda. **Keyword:** `RGB pixel flatten reshape -1 3`, `KMeans color quantization sampling pixels`, `numpy sample rows rgb`.

## 10. K-Means segmentasi warna dan reshape kembali

In [ ]:
km_warna = KMeans(n_clusters=4, random_state=42, n_init=10)
km_warna.fit(X_pixel)
label_semua_pixel = km_warna.predict(pixel_flat)
warna_segmentasi = km_warna.cluster_centers_[label_semua_pixel].astype('uint8')
gambar_segmentasi = warna_segmentasi.reshape(rgb.shape)
print('Shape prediksi semua pixel:', label_semua_pixel.shape)
print('Shape warna dari cluster:', warna_segmentasi.shape)
print('Shape sesudah reshape kembali:', gambar_segmentasi.shape)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(rgb)
ax[0].set_title('Asli')
ax[1].imshow(gambar_segmentasi)
ax[1].set_title('Hasil K-Means 4 warna')
for a in ax: a.axis('off')
plt.tight_layout()
plt.show()

**Interpretasi:** Cluster warna diwakili **centroid RGB**; hasil `predict` dipetakan lagi menjadi citra berdimensi asli. Jadi **clustering pixel** di sini berfungsi untuk segmentasi/penyederhanaan warna, bukan clustering antargambar. **Keyword:** `KMeans image segmentation color quantization`, `KMeans predict all pixels reshape`, `kmeans centroid RGB image`.

## 11. Ringkasan 2 tugas image

In [ ]:
print('A — Clustering antar gambar:')
print('    gambar_sample:', gambar_sample.shape, '-> X_digit:', X_digit.shape)
print('B — Segmentasi warna pixel:')
print('    rgb:', rgb.shape, '-> pixel_flat:', pixel_flat.shape, '-> hasil:', gambar_segmentasi.shape)
print('Selesai praktikum image.')

**Interpretasi:** Jika soal menyebut **kelompok foto**: buat fitur per foto `(n_foto, n_fitur)`. Jika soal menyebut **segmentasi warna dalam satu foto**: buat fitur per pixel `(n_pixel, 3)`. **Keyword:** `image clustering vs pixel clustering`, `how to choose samples and features image kmeans`.